In [77]:
from openai import OpenAI
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback
from ingest import load_lessons_data
from gitsource import chunk_documents
from minsearch import Index
import os
from dotenv import load_dotenv
from groq import Groq
load_dotenv()

INSTRUCTIONS = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

PROMPT_TEMPLATE = """
QUESTION: {question}

CONTEXT:
{context}
""".strip()

In [41]:
documents = load_lessons_data()

In [67]:
chunks = chunk_documents(
    documents,
    size=2000,
    step=1000
)

chunk_index = Index(
    text_fields = ["content"],
    keyword_fields = ["filename"]
)
chunk_index.fit(chunks)

In [68]:
tools = Tools()

In [69]:
search_calls = 0

def search(query: str) -> dict[str, str]:
    """
    Search the course lessons and return relevant passages.
    """
    global search_calls
    search_calls += 1

    return chunk_index.search(
        query=query,
        num_results=3
    )

In [70]:
tools.add_tool(search)

In [71]:
print(tools.__dict__)

{'tools': {'search': {'type': 'function', 'name': 'search', 'description': 'Search the course lessons and return relevant passages.', 'parameters': {'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'query parameter'}}, 'required': ['query'], 'additionalProperties': False}}}, 'functions': {'search': <function search at 0x000001E9A2C1E5C0>}}


In [72]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [ ]:
groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))



# # Create and run chat assistant
runner = OpenAIResponsesRunner(
    tools=tools,
    developer_prompt=INSTRUCTIONS,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(
        client=groq_client,
        model="openai/gpt-oss-120b"
    )
)

In [83]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

AttributeError: 'Groq' object has no attribute 'responses'

In [87]:
llm_client=OpenAIClient(
    client=groq_client,
    model="openai/gpt-oss-120b"
)
print(llm_client.__dict__)

{'model': 'openai/gpt-oss-120b', 'client': <groq.Groq object at 0x000001E9A65FC850>, 'extra_kwargs': {}}


In [25]:
result.cost

In [26]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How does the agentic loop work, and how is it different from plain RAG?', role='user', phase=None, type=None),
 ResponseReasoningItem(id='resp_01kvjedmpsfzw9paty9dw5eh7g', summary=[], type='reasoning', content=None, encrypted_content=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"agentic loop vs plain RAG"}', call_id='zjj997vr9', name='search', type='function_call', id='zjj997vr9', namespace=None, status='c

In [27]:
result2 = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    previous_messages=result.all_messages,
    callback=callback,
)

BadRequestError: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=search({"query": "agentic loop vs plain RAG"})</function>'}}

In [37]:
print(search_calls)

0


In [17]:
print(tools.get_definitions())

AttributeError: 'Tools' object has no attribute 'get_definitions'

In [18]:
models = groq_client.models.list()

for m in models.data:
    print(m.id)

meta-llama/llama-prompt-guard-2-86m
whisper-large-v3-turbo
llama-3.3-70b-versatile
allam-2-7b
openai/gpt-oss-120b
qwen/qwen3.6-27b
openai/gpt-oss-safeguard-20b
llama-3.1-8b-instant
canopylabs/orpheus-v1-english
qwen/qwen3-32b
meta-llama/llama-4-scout-17b-16e-instruct
openai/gpt-oss-20b
whisper-large-v3
canopylabs/orpheus-arabic-saudi
groq/compound-mini
meta-llama/llama-prompt-guard-2-22m
groq/compound


In [28]:
tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the course lessons and return relevant passages.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [58]:
from groq import Groq

client = Groq()
completion = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
      {
        "role": "user",
        "content": "Hi"
      },
   ]
    # temperature=1,
    # max_completion_tokens=8192,
    # top_p=1,
    # reasoning_effort="medium",
    # stream=True,
    # stop=None
)

In [65]:
from groq import Groq

client = Groq()

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role": "user", "content": "Explain Python lists"}
    ],
)

print(response.choices[0].message.content)

## Python Lists – A Comprehensive Overview

A **list** is one of the most frequently used built‑in data structures in Python.  
Think of it as a dynamic array that can store an ordered collection of items of **any** type—numbers, strings, objects, even other lists.

---

## 1️⃣ Core Characteristics

| Feature | Description |
|---------|-------------|
| **Ordered** | Elements keep the order you insert them. `list[0]` is always the first item. |
| **Mutable** | You can change, add, or delete items after the list has been created. |
| **Heterogeneous** | A single list can hold values of different types (`[1, "two", 3.0]`). |
| **Dynamic sizing** | No need to pre‑declare a size; it grows/shrinks automatically. |
| **Zero‑based indexing** | The first element is at index `0`, the last at `-1`. |

---

## 2️⃣ Creating Lists

```python
# Empty list
empty = []

# From literals
primes = [2, 3, 5, 7, 11]

# From an iterable (e.g., range, string, another list)
nums = list(range(5))        # [0, 1,